# Autoencoders — Unsupervised Feature Learning & Generative Models

- **Vanilla Autoencoder**: Compress → Reconstruct
- **Denoising Autoencoder**: Learn to remove noise
- **Variational Autoencoder (VAE)**: Generate new data
- **Applications**: Dimensionality reduction, anomaly detection, image generation

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.datasets import mnist, fashion_mnist
import warnings
warnings.filterwarnings('ignore')
print(f'TensorFlow: {tf.__version__}')

In [ ]:
# Load MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0
X_train_flat = X_train.reshape(-1, 784)
X_test_flat = X_test.reshape(-1, 784)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 1. Vanilla Autoencoder

```
Input (784) → Encoder → Latent (32) → Decoder → Output (784)
```

The network learns to compress and reconstruct. The **bottleneck** (latent space) forces it to learn the most important features.

In [ ]:
# Build Autoencoder
latent_dim = 32

# Encoder
encoder_input = keras.Input(shape=(784,))
x = layers.Dense(256, activation='relu')(encoder_input)
x = layers.Dense(128, activation='relu')(x)
encoded = layers.Dense(latent_dim, activation='relu')(x)
encoder = Model(encoder_input, encoded, name='encoder')

# Decoder
decoder_input = keras.Input(shape=(latent_dim,))
x = layers.Dense(128, activation='relu')(decoder_input)
x = layers.Dense(256, activation='relu')(x)
decoded = layers.Dense(784, activation='sigmoid')(x)
decoder = Model(decoder_input, decoded, name='decoder')

# Autoencoder
autoencoder = Model(encoder_input, decoder(encoder(encoder_input)), name='autoencoder')
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

history = autoencoder.fit(X_train_flat, X_train_flat, epochs=20, batch_size=256,
                          validation_data=(X_test_flat, X_test_flat), verbose=0)
print(f'Final Loss — Train: {history.history["loss"][-1]:.4f} | Val: {history.history["val_loss"][-1]:.4f}')

In [ ]:
# Visualize reconstructions
reconstructed = autoencoder.predict(X_test_flat[:10], verbose=0)

fig, axes = plt.subplots(2, 10, figsize=(16, 3))
for i in range(10):
    axes[0, i].imshow(X_test[i], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(reconstructed[i].reshape(28, 28), cmap='gray'); axes[1, i].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=11)
axes[1, 0].set_ylabel('Reconstructed', fontsize=11)
plt.suptitle(f'Vanilla Autoencoder — Latent Dim: {latent_dim}', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize latent space with t-SNE
from sklearn.manifold import TSNE

encoded_imgs = encoder.predict(X_test_flat[:3000], verbose=0)
tsne = TSNE(n_components=2, random_state=42)
encoded_2d = tsne.fit_transform(encoded_imgs)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(encoded_2d[:, 0], encoded_2d[:, 1], c=y_test[:3000], cmap='tab10', s=5, alpha=0.6)
plt.colorbar(scatter, label='Digit')
plt.title('Autoencoder Latent Space (t-SNE projection)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Denoising Autoencoder

Add noise to inputs → train to reconstruct clean outputs. Forces learning of robust features.

In [ ]:
# Add Gaussian noise
noise_factor = 0.5
X_train_noisy = X_train_flat + noise_factor * np.random.normal(size=X_train_flat.shape)
X_test_noisy = X_test_flat + noise_factor * np.random.normal(size=X_test_flat.shape)
X_train_noisy = np.clip(X_train_noisy, 0, 1)
X_test_noisy = np.clip(X_test_noisy, 0, 1)

# Build denoising autoencoder (same architecture)
denoise_ae = keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(32, activation='relu'),  # bottleneck
    layers.Dense(128, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dense(784, activation='sigmoid')
])
denoise_ae.compile(optimizer='adam', loss='binary_crossentropy')

# Train: noisy input → clean target
denoise_ae.fit(X_train_noisy, X_train_flat, epochs=20, batch_size=256,
               validation_data=(X_test_noisy, X_test_flat), verbose=0)

# Denoise test images
denoised = denoise_ae.predict(X_test_noisy[:10], verbose=0)

fig, axes = plt.subplots(3, 10, figsize=(16, 5))
for i in range(10):
    axes[0, i].imshow(X_test[i], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(X_test_noisy[i].reshape(28, 28), cmap='gray'); axes[1, i].axis('off')
    axes[2, i].imshow(denoised[i].reshape(28, 28), cmap='gray'); axes[2, i].axis('off')
axes[0, 0].set_ylabel('Original'); axes[1, 0].set_ylabel('Noisy'); axes[2, 0].set_ylabel('Denoised')
plt.suptitle('Denoising Autoencoder', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Variational Autoencoder (VAE)

Unlike vanilla AE, VAE learns a **probability distribution** in latent space, enabling **generation** of new data.

```
Input → Encoder → (μ, σ) → Sample z ~ N(μ, σ) → Decoder → Output
```

Loss = Reconstruction Loss + KL Divergence

In [ ]:
# VAE
latent_dim_vae = 2  # 2D for visualization

# Encoder
enc_input = keras.Input(shape=(784,))
x = layers.Dense(256, activation='relu')(enc_input)
x = layers.Dense(128, activation='relu')(x)
z_mean = layers.Dense(latent_dim_vae, name='z_mean')(x)
z_log_var = layers.Dense(latent_dim_vae, name='z_log_var')(x)

# Reparameterization trick
def sampling(args):
    z_mean, z_log_var = args
    epsilon = K.random_normal(shape=(K.shape(z_mean)[0], latent_dim_vae))
    return z_mean + K.exp(0.5 * z_log_var) * epsilon

z = layers.Lambda(sampling)([z_mean, z_log_var])
vae_encoder = Model(enc_input, [z_mean, z_log_var, z], name='vae_encoder')

# Decoder
dec_input = keras.Input(shape=(latent_dim_vae,))
x = layers.Dense(128, activation='relu')(dec_input)
x = layers.Dense(256, activation='relu')(x)
dec_output = layers.Dense(784, activation='sigmoid')(x)
vae_decoder = Model(dec_input, dec_output, name='vae_decoder')

# VAE model
vae_output = vae_decoder(z)
vae = Model(enc_input, vae_output, name='vae')

# VAE loss
reconstruction_loss = keras.losses.binary_crossentropy(enc_input, vae_output) * 784
kl_loss = -0.5 * K.sum(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var), axis=-1)
vae_loss = K.mean(reconstruction_loss + kl_loss)
vae.add_loss(vae_loss)
vae.compile(optimizer='adam')

vae.fit(X_train_flat, epochs=30, batch_size=256,
        validation_data=(X_test_flat, None), verbose=0)
print('VAE training complete!')

In [ ]:
# Visualize VAE latent space (2D)
z_mean_test, _, _ = vae_encoder.predict(X_test_flat, verbose=0)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(z_mean_test[:, 0], z_mean_test[:, 1], c=y_test, cmap='tab10', s=5, alpha=0.5)
plt.colorbar(scatter, label='Digit')
plt.title('VAE Latent Space (2D) — Digits')
plt.xlabel('z[0]'); plt.ylabel('z[1]')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Generate new digits by sampling the latent space
n = 15
grid_x = np.linspace(-3, 3, n)
grid_y = np.linspace(-3, 3, n)

figure = np.zeros((28 * n, 28 * n))
for i, yi in enumerate(grid_y):
    for j, xi in enumerate(grid_x):
        z_sample = np.array([[xi, yi]])
        x_decoded = vae_decoder.predict(z_sample, verbose=0)
        digit = x_decoded[0].reshape(28, 28)
        figure[i * 28:(i + 1) * 28, j * 28:(j + 1) * 28] = digit

plt.figure(figsize=(10, 10))
plt.imshow(figure, cmap='gray')
plt.title('VAE — Generated Digits (Latent Space Grid)', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

## Summary

| Type | Latent Space | Generative? | Best For |
|------|-------------|-------------|----------|
| **Vanilla AE** | Deterministic | No | Compression, denoising |
| **Denoising AE** | Deterministic | No | Robust features, noise removal |
| **VAE** | Probabilistic (μ, σ) | **Yes** | Generating new data |
| **Convolutional AE** | Deterministic | No | Image compression |

### Applications:
- **Anomaly detection**: High reconstruction error = anomaly
- **Image denoising/super-resolution**
- **Feature extraction** for downstream tasks
- **Data generation** (VAE, next: GANs)